# GROBID PDF Parsing + Pinecone Embedding

In [1]:
from langchain_community.document_loaders.parsers import GrobidParser
from langchain_community.document_loaders.generic import GenericLoader

/Users/a215482/Desktop/personal projects/rag-citation-langchain/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
try:
    parser = GrobidParser(segment_sentences=False)
    print("Parser created")
    print(parser)
except Exception as e:
    print(f"Error: {e}")


Parser created


In [3]:

loader = GenericLoader.from_filesystem(
    ".",
    glob="hope.pdf",
    suffixes=[".pdf"],
    parser=GrobidParser(segment_sentences=False)
)

In [4]:
try :
    loader.load()
except Exception as e:
    print(f"Error: {e}")


In [5]:
docs = loader.load()

In [6]:
docs[0].metadata

{'text': "We follow Baron's (2008) seminal work in defining the relevant terms here.Affect represents the emotions, moods, and passions of the entrepreneurit is the realized subjective feelings of (dis)pleasure attached to an object (e.g., experience; physical thing), where such feelings that can differ in length, intensity, antecedent, and so on.The entrepreneurial process consists of creatively identifying an opportunity and acting to exploit it, all under uncertainty (where the opportunity can be identified ex post as a market imperfection believed ex ante to indicate that unrealizable value existed from which some party can profit).That process is often characterized by multiple stages, non-routineness, and imperfection.Because such characterizations are associated with higher levels of emotion (e.g., where the personal stakes and pressures are greater -Cardon et al., 2012), the entrepreneurial process requires attention be paid to affect.Baron (2008) highlights the many potential 

In [7]:
from langchain_text_splitters import CharacterTextSplitter
from models import Chunk

In [8]:
chunks: list[Chunk] = []
for doc in docs:
    text_splitter = CharacterTextSplitter.from_tiktoken_encoder(
        encoding_name="cl100k_base", chunk_size=1500, chunk_overlap=300
    )
    texts = text_splitter.split_text(doc.page_content)
    for i, text in enumerate(texts):
        chunk = Chunk(
            text=text,
            paragraph=doc.metadata["para"],
            pages=doc.metadata["pages"],
            section_title=doc.metadata["section_title"],
            section_number=doc.metadata["section_number"],
            paper_title=doc.metadata["paper_title"],
        )
        chunks.append(chunk)

chunks


[Chunk(text="We follow Baron's (2008) seminal work in defining the relevant terms here.Affect represents the emotions, moods, and passions of the entrepreneurit is the realized subjective feelings of (dis)pleasure attached to an object (e.g., experience; physical thing), where such feelings that can differ in length, intensity, antecedent, and so on.The entrepreneurial process consists of creatively identifying an opportunity and acting to exploit it, all under uncertainty (where the opportunity can be identified ex post as a market imperfection believed ex ante to indicate that unrealizable value existed from which some party can profit).That process is often characterized by multiple stages, non-routineness, and imperfection.Because such characterizations are associated with higher levels of emotion (e.g., where the personal stakes and pressures are greater -Cardon et al., 2012), the entrepreneurial process requires attention be paid to affect.Baron (2008) highlights the many potenti

In [9]:
import voyageai
from dotenv import load_dotenv
from typing import List

load_dotenv()

def get_voyage_client():

    return voyageai.Client()

In [10]:
vo = get_voyage_client()

In [12]:
from models import Embedding 

embeddingsList: List[Embedding] = []
inputs: List[str] = []
for chunk in chunks:
    inputs.append(chunk.text)


# Get embeddings from Voyage AI
raw_embeddings = vo.embed(
    texts=inputs, model="voyage-4-lite", input_type="document"
)

for i, raw_embedding in enumerate(raw_embeddings.embeddings):
    embedding = Embedding(
        chunk=chunks[i],
        embedding=raw_embedding
    )
    embeddingsList.append(embedding)


In [19]:
from pinecone import Pinecone
from dotenv import load_dotenv



load_dotenv()

def get_pinecone_client():
    return Pinecone()

pc = get_pinecone_client()

In [21]:
from pinecone import ServerlessSpec

index_name = "langchain-test-index"

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=1536,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )

index = pc.Index(index_name)

In [27]:
from langchain_pinecone import PineconeVectorStore

vector_store = PineconeVectorStore(
    index=index,
    embedding=embeddingsList
)
